# Week 1 Day 5 Lab 2 - Harness Engineering

## Deep Agents works the queue on its own

Deep Agents is LangChain's ready-made harness. It's built on `create_agent`, the same function we used on Day 2, with middleware added for file tools, subagents, a to-do list and summarizing a long conversation. It runs on any model LangChain supports, so we can use OpenRouter.

This is the same job as Lab 1: a folder with five claims from the queue, the policy wording and the claims manual, and the instruction to decide every claim. We don't write any tools.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls DeepSeek V4.1 Flash through OpenRouter, and a run on five claims costs about 3 cents.
            If you'd rather not spend anything, there's a free version of this lab in <a href="week1day5_lab2free.ipynb">week1day5_lab2free.ipynb</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_openai import ChatOpenAI

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Part 1: The claims desk

The claims desk is the same as in Lab 1. We use the five claims marked for the tiny version of the queue, which include one that needs the claims manual.

In [13]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=True):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [14]:
claims, labels = get_sample_data(tiny=False)

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1101 Typhoon canceled a flight on a round the world trip, 26 hour delay
CLM-1102 Laptop bag stolen from a parked rental car
CLM-1103 Suitcase and clothes ruined by fish sauce in the hold on the way to Hanoi
CLM-1104 Dentures lost overboard during rough seas
CLM-1105 Trek to Machu Picchu canceled after breaking an ankle at pickleball
CLM-1106 Action camera stolen by a kea at Arthur's Pass
CLM-1107 Four hour delay leaving Houston because of a swarm of bees on the wing
CLM-1108 Seven hour delay on the train from Portland to Los Angeles
CLM-1109 Business trip to Transylvania cut short by sickness
CLM-1110 Wallet and phone pickpocketed on the Paris metro
CLM-1111 Decompression sickness after a scuba dive, hyperbaric treatment
CLM-1112 Phone, earbuds and speaker ruined when a kayak capsized in Krabi
CLM-1113 Eleven hour weather delay leaving Denver, meals paid in cash
CLM-1114 Walking trip to England canceled after the town demolished his house for a bypass
CLM-1115 Emergency room visit 

In [15]:
DESK = Path("lab2_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

## Part 2: Setting up the harness

The instructions are the same as in Lab 1, apart from calling the task list a to-do list, which is Deep Agents' name for it.

In [16]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Deep Agents</h2>
            <span style="color:#118AB2;"><code>create_deep_agent</code> gives the agent file tools (<code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>) and <code>task</code>, which hands work to a subagent.<br/><br/>
            The to-do list is an extra since version 0.7, so we add it with LangChain's <code>TodoListMiddleware</code>, which gives the agent a <code>write_todos</code> tool.<br/><br/>
            <code>FilesystemBackend</code> points the file tools at the claims desk. With <code>virtual_mode=True</code> the agent sees the desk as <code>/</code> and can't reach anything outside it.
            </span>
        </td>
    </tr>
</table>

In [17]:
llm = ChatOpenAI(
    model="deepseek/deepseek-v4.1-flash",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort="low",
    timeout=120,
    max_retries=5,
)

In [18]:
claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware()],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)

## Part 3: Working the queue

As the agent works we print each step and count the model calls and the cost. The stream gives us an update after each step of the agent, and the updates from the `model` step hold the model's replies. The subagent's steps are indented.

In [19]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call["name"]

In [20]:
cost = 0
calls = 0

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            cost += message.response_metadata["token_usage"]["cost"]
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

print(f"\nFinished after {calls} model calls for ${cost:.4f}")

ls /
ls /claims/
read_file /claims_manual.md
read_file /company.json
read_file /policy_wording.md
read_file /claims_manual.md
read_file /policy_wording.md
ls /decisions/
read_file /claims/CLM-1101.json
read_file /claims/CLM-1102.json
read_file /claims/CLM-1103.json
read_file /claims/CLM-1104.json
read_file /claims/CLM-1105.json
ls /policies/
ls /customers/
write_todos: CLM-1101 trip delay (Fogg) (in_progress), CLM-1102 baggage theft from re (in_progress), CLM-1103 baggage damage fish s (in_progress), CLM-1104 dentures lost overboa (in_progress), CLM-1105 trip cancellation bro (in_progress), CLM-1106 (pending), CLM-1107 (pending), CLM-1108 (pending), CLM-1109 (pending), CLM-1110 (pending), CLM-1111 (pending), CLM-1112 (pending), CLM-1113 (pending), CLM-1114 (pending), CLM-1115 (pending), CLM-1116 (pending), CLM-1117 (pending), CLM-1118 (pending), CLM-1119 (pending), CLM-1120 (pending), CLM-1121 (pending), CLM-1122 (pending), CLM-1123 (pending), CLM-1124 (pending), CLM-1125 (pending), Re

## Part 4: How it did

We score the decision files with the same test as in Lab 1.

In [21]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(float(answer.get("payout") or 0) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

24 of 25 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1101,approve,580.0,approve,580.0,True
1,CLM-1102,decline,0.0,decline,0.0,True
2,CLM-1103,approve,956.0,approve,956.0,True
3,CLM-1104,approve,250.0,approve,250.0,True
4,CLM-1105,approve,2400.0,approve,2400.0,True
5,CLM-1106,approve,250.0,approve,250.0,True
6,CLM-1107,decline,0.0,decline,0.0,True
7,CLM-1108,approve,66.0,refer,0.0,False
8,CLM-1109,approve,890.0,approve,890.0,True
9,CLM-1110,refer,0.0,refer,0.0,True


And here's the note it left for the handler.

In [22]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Handover — claims queue worked (25 claims, CLM-1101 to CLM-1125)

All 25 open claims have a decision file in `decisions/`. Decided under policy wording 2026.1 and claims manual 2026.2.

**Summary:** 15 approved ($15,215.00 total), 5 declined, 5 referred.

## Approved (15)

| Claim | Customer | Benefit | Payout | Note |
|---|---|---|---|---|
| CLM-1101 | Phil Fogg | Trip Delay | $580.00 | First. Hotel $310 + meals $120 + rebooking $150. $2,000 forfeited wager excluded as consequential loss (s.9). |
| CLM-1103 | Dennis Kaminski | Baggage | $956.00 | Premium. Sweaters capped at $300/item; less $450 Northwind settlement and $50 deductible. |
| CLM-1104 | Walter Ferris | Baggage | $250.00 | Premium. Dentures = one item, capped $300, less $50 deductible. |
| CLM-1105 | Daniel Cho | Trip Cancellation | $2,400.00 | **Needs sign-off.** Broken ankle, non-refundable tour, capped at trip cost. |
| CLM-1106 | Colin Meadows | Baggage | $250.00 | Premium. Kea took camera = accidental loss, no police report needed; $300 cap less $50. |
| CLM-1108 | Marjorie Keane | Trip Delay | $66.00 | Premium. Locomotive failure, 7h25m, receipted meals. |
| CLM-1109 | Jonathan Harker | Trip Interruption | $890.00 | Premium. Fare home $610 + unused nights $280. |
| CLM-1111 | Leilani Kahale | Emergency Medical | $4,600.00 | **Needs sign-off.** DCS on scuba; Adventure Sports add-on held, certification and depth in limits. |
| CLM-1112 | Mia Castellano | Baggage | $680.00 | Economy + Electronics Plus. Only the phone gets the $1,200 limit; earbuds and speaker capped at $150 each; less $100 deductible. |
| CLM-1113 | Grace Liu | Trip Delay | $259.00 | First. Hotel $189 + meals without receipts at $35 x 2 travellers x 1 day = $70. |
| CLM-1115 | Harold Benson | Emergency Medical | $730.00 | Premium + waiver. $780 less $50 deductible. |
| CLM-1117 | Trevor Mallory | Emergency Medical | $780.00 | Economy. $880 less $100 deductible. |
| CLM-1118 | Julian Okoro | Baggage | $1,880.00 | **Needs sign-off.** First: suit and camera+lens each one item at $500; less $500 carrier settlement. |
| CLM-1120 | Lorraine Baptiste | Trip Delay | $404.00 | Premium. Return-journey delay, cover extends; $690 suite reduced to the $350 a night rate + $54 dinner. |
| CLM-1121 | Philip Sherman | Baggage | $490.00 | Premium. Mask $260 + camera $280, less $50 deductible. |

**Three payouts exceed $1,000 and need claims-handler sign-off before payment** (CLM-1105 $2,400; CLM-1111 $4,600; CLM-1118 $1,880 — $8,880 in all). They are recorded as approve with the payout and a `sign_off` block; payment waits. I have not signed them; whoever releases the money should check the decision and the calculation and record their name against the payment.

## Declined (5)

- **CLM-1102** Tyler Brooks, $0 — laptop bag left on the back seat of a locked car; unattended belongings, wording s.5 (trunk or glove compartment only).
- **CLM-1107** Gloria Esposito, $0 — 4h20m delay caused by a swarm of bees, which is not one of the causes listed in Trip Delay, s.3. The wild-animal note in the manual covers baggage, not delay.
- **CLM-1114** Arthur Dent, $0 — house demolished for a bypass is not "fire, flood or storm"; s.2 covers no other reason.
- **CLM-1119** Jake Morrison, $0 — shoulder dislocated bungee jumping; adventure sports excluded under s.7 and no add-on on the Economy plan.
- **CLM-1124** Alfred Linguini, $0 — knife roll and cookbook chewed by a rat; vermin exclusion in s.5, manual confirms rats and mice stay excluded.

## Referred (5)

**Awaiting a document (claim open 30 days):**
- **CLM-1110** Amara Osei — theft on the Paris metro is covered and looks payable at $605, but only a police reference number is on file. Needs the police report for reference 2026/PV/093317.
- **CLM-1123** Douglas Pruitt — 11-hour ferry delay, appears payable at $89, but the Aegean Star certificate gives no cause. Needs the carrier's written confirmation of cause.

**To the Special Investigations Unit:**
- **CLM-1116** Marisol Vega — same June 9 Cancun clinic visit already claimed and paid under CLM-0979 ($320). Possible duplicate.
- **CLM-1122** Carla Jennings — Baggage Delay receipts are dated August 7 in Columbus, three days before the delay and at home. Looks payable at $100 if the purchases are genuine delay expenses.
- **CLM-1125** Nathan Brandt — third claim in 12 months (CLM-0899, CLM-0968, this one). Appears payable at the Economy Baggage Delay maximum of $100.

Customers have been told only that referred claims are under review where the referral is to SIU — no accusation of dishonesty in any file or letter.

## Things worth knowing

- **Plan matters more than the story.** Several payouts turn on the waiting period or per-item limit; Economy vs First changed the answer in CLM-1113, CLM-1118 and CLM-1120.
- **Sets and pairs.** Applied as one item in CLM-1104 (dentures), CLM-1103 (boots), CLM-1118 (suit; camera with lens).
- **Electronics Plus is narrow.** It names phones, tablets, laptops, cameras, e-readers and smartwatches only — earbuds and speakers stayed at the plan limit in CLM-1112.
- **Wild animals vs vermin.** Kea (CLM-1106) covered as accidental loss; rat (CLM-1124) excluded. Keep that distinction.
- **Two odd-looking claims are genuine and paid:** the kea strike and the dentures overboard. Unusual is not the same as untrue.
- Every decision file carries the full record required by manual s.14: decision, payout, wording section, calculation, plain-language reason, wording and manual version, and who decided and when.
- No customer's medical details or personal data appear in any other customer's file.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">Deep Agents is open source and runs on any model, so you can pick a cheap one and read exactly what the harness does. The Claude Agent SDK gives you Claude Code's harness, tuned for Claude models.<br/><br/>
            Next we build the same pieces into UnderCover ourselves: a to-do list, tools to read the manual, and a subagent for each claim.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its to-do list and where it handed work to the subagent.<br/><br/>
            Then try other models and compare accuracy, cost and time: <code>openai/gpt-6-luna</code>, or <code>anthropic/claude-sonnet-5.5</code> to compare with Lab 1 on the same model, which costs more.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>